# Field baselines

> How much of the Cs-137 ratio and Kex is a property of the field, and whether the spectra beat what a field's own wet chemistry, its soil type and land use, or its neighbours predict

In [ ]:
#| default_exp baselines

In [ ]:
#| export
import numpy as np, pandas as pd
from sklearn.neighbors import NearestNeighbors

In [ ]:
import warnings
import numpy as np, pandas as pd
from scipy.linalg import LinAlgWarning
from sklearn.linear_model import Ridge
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from fastcore.test import *
from soilspecdl.all import group_folds, regression_scores
from ossl_fuku_transfer.data import *
from ossl_fuku_transfer.transfer import preprocessings, load_pretrained
from ossl_fuku_transfer.grouped import oof_preds, sample_scores
from ossl_fuku_transfer.inside import layer_features
warnings.simplefilter('ignore', LinAlgWarning)

/Users/franckalbinet/aai-ws/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


The cells that read the Fukushima data are marked `#| fukudata`. The outputs on this page come from a local run.

Remediation decisions are made per field, or per group of fields. Two uses of the spectra follow:

- Fields with some wet chemistry: a few samples of a field are measured, and the spectra predict the rest. Here the spectra must beat the field's own measured samples.
- Fields with no wet chemistry: the spectra predict fields that have no measured samples. Here the spectra must beat what is known without them, such as the field's soil type and land use, or the values of neighbouring fields.

The earlier notebooks scored the spectra against random or field-grouped splits, without these baselines. This notebook adds them, for the Cs-137 ratio of `10_cs137` (log10 of exchangeable over total Cs-137) and `log1p(Kex)`.

## Data

The current selection, with the field, sampling year, soil type, land use and coordinates of each sample, the spectra after SNV and the first derivative, and block 4 of the OSSL Kex and CEC models, as in `20_grouped_cv`:

In [ ]:
#| fukudata
fuku = load_fuku()
ids = fuku.sample_ids
src = fuku.source.reindex(ids)
ratio = pd.concat([pd.read_csv(fuku_path()/'reference'/f'cs137_ratio_{s}.csv', usecols=['ID', 'exCs137_totalCs137']) for s in ('train', 'test')])
ratio = ratio.assign(ID=ratio.ID.astype(str)).set_index('ID')['exCs137_totalCs137']
targets = {'Cs-137 ratio': ratio.reindex(ids).to_numpy(), 'Kex': np.log1p(src['soil_ex_K2O'].to_numpy(dtype=float) / mg_k2o_per_cmolc)}
fields, years = src['field_no'].astype(str).to_numpy(), src['year'].to_numpy()
cat = src[['Soil_type_describe', 'fieldtype']].fillna('unknown').astype(str).reset_index(drop=True)
coords = src[['Field_place(latitude)', 'Field_place(longitude)']].apply(pd.to_numeric, errors='coerce').to_numpy()
Xd = preprocessings()['snv_d1'].fit_transform(fuku.spectra)
B4 = np.hstack([layer_features(load_pretrained(n), Xd)['block 4'] for n in ('ossl_kex_snv_d1', 'ossl_cec_snv_d1')])
len(ids), len(np.unique(fields))

(1389, 391)

Each field has almost one sample per sampling year, so the spread within a field in one campaign cannot be measured here. The spread within a field is across years, from 2013 to 2020:

In [ ]:
#| fukudata
fy = pd.Series(1, index=pd.MultiIndex.from_arrays([fields, years])).groupby(level=[0, 1]).size()
pd.DataFrame({'samples per field and year': fy.value_counts().sort_index(),
              'samples per field': pd.Series(fields).value_counts().value_counts().sort_index()}).fillna(0).astype(int)

,samples per field and year,samples per field
1,1377,122
2,6,55
3,0,20
4,0,36
5,0,53
6,0,62
7,0,28
8,0,15


The coordinates set how precise the neighbour baseline below can be. The number of distinct coordinates against the number of fields:

In [ ]:
#| fukudata
cp = pd.DataFrame(coords, columns=['lat', 'lon']).assign(field=fields)
pd.Series({'fields': cp.field.nunique(), 'distinct coordinates': len(cp[['lat', 'lon']].drop_duplicates()),
           'fields with more than one coordinate': int((cp.groupby('field')[['lat', 'lon']].nunique().max(axis=1) > 1).sum())})

fields                                  391
distinct coordinates                    386
fields with more than one coordinate      2
dtype: int64

## Variance between and within fields

`field_variance` splits the variance of a property into a part between fields and a part within fields, by one-way random-effects analysis of variance, on the fields with at least two samples. The share of the sampling year is the variance of the year means over the total variance:

In [ ]:
#| export
def field_variance(
    y, # Property of each sample
    g, # Field of each sample
)->dict: # Number of fields and samples, standard deviations between and within fields, and the share between
    "Method-of-moments split of the variance of `y` between and within the groups `g` that have at least two values"
    d = pd.DataFrame({'y': y, 'g': g}).dropna()
    d = d[d.groupby('g').y.transform('size') >= 2]
    n, N, k = d.groupby('g').size(), len(d), d.g.nunique()
    msw = ((d.y - d.groupby('g').y.transform('mean'))**2).sum() / (N - k)
    msb = (n * (d.groupby('g').y.mean() - d.y.mean())**2).sum() / (k - 1)
    sb = max((msb - msw) / ((N - (n**2).sum() / N) / (k - 1)), 0)
    return {'fields': k, 'samples': N, 'sd between': np.sqrt(sb), 'sd within': np.sqrt(msw), 'share between': sb / (sb + msw)}

In [ ]:
rng = np.random.default_rng(0)
g_ = np.repeat(np.arange(200), 5)
y_ = rng.normal(0, 2, 200)[g_] + rng.normal(0, 1, 1000)
v_ = field_variance(y_, g_)
test_close(v_['share between'], 0.8, eps=0.06)
test_close(v_['sd within'], 1, eps=0.06)

In [ ]:
#| fukudata
def year_share(y): return pd.Series(y).groupby(years).mean().var() / np.nanvar(y)
pd.DataFrame({t: {**field_variance(y - pd.Series(y).groupby(years).transform('mean').to_numpy(), fields), 'share of the years': year_share(y)}
              for t, y in targets.items()}).T.round(3)

,fields,samples,sd between,sd within,share between,share of the years
Cs-137 ratio,269.0,1267.0,0.322,0.132,0.855,0.029
Kex,269.0,1266.0,0.203,0.115,0.759,0.006


The same split, with the year means removed, by land use:

In [ ]:
#| fukudata
pd.DataFrame({(t, ft): field_variance((y - pd.Series(y).groupby(years).transform('mean').to_numpy())[m], fields[m])['share between']
              for t, y in targets.items() for ft in ['paddy_field', 'upland_field'] for m in [cat.fieldtype.to_numpy() == ft]}, index=['share between']).T.round(3)

share between
Cs-137 ratio paddy_field           0.849
             upland_field          0.855
Kex          paddy_field           0.662
             upland_field          0.739

## Fields with wet chemistry

Each sample is predicted by the mean of the other samples of its field, all from other years, and by one other sample of its field drawn at random. These are what a field's own wet chemistry gives. The spectra are scored with shuffled folds, so their test samples also come from fields with training samples, and with the field mean as a starting point, by a ridge fitted to the residual from it:

In [ ]:
#| fukudata
ridge = (make_pipeline(StandardScaler(), Ridge()), {'ridge__alpha': np.logspace(-1, 6, 15)})
sid = np.asarray(ids)
res = {}
for t, y in targets.items():
    ok = np.isfinite(y)
    d = pd.DataFrame({'f': fields[ok], 'y': y[ok]})
    g = d.groupby('f').y
    n = g.transform('size').to_numpy()
    loo = ((g.transform('sum') - d.y) / (g.transform('size') - 1)).to_numpy()
    one = g.transform(lambda s: [np.random.default_rng(0).choice(np.delete(s.values, i)) if len(s) > 1 else np.nan for i in range(len(s))]).to_numpy()
    has = n >= 2
    yy, Xh, sh = y[ok][has], Xd[ok][has], sid[ok][has]
    spec, _ = oof_preds(*ridge, Xh, yy, group_folds(sh), sh)
    resid, _ = oof_preds(*ridge, Xh, yy - loo[has], group_folds(sh), sh)
    P = {'mean of the field\'s other samples': loo[has], 'one other sample of the field': one[has],
         'ridge, spectra, shuffled folds': spec, 'field mean + ridge on its residual': loo[has] + resid}
    res[t] = pd.DataFrame({k: regression_scores(yy, p) for k, p in P.items()}).T[['R²', 'RMSE']].assign(samples=len(yy))
pd.concat(res).round(3)

R²   RMSE  samples
Cs-137 ratio mean of the field's other samples   0.813  0.152     1267
             one other sample of the field       0.678  0.199     1267
             ridge, spectra, shuffled folds      0.584  0.227     1267
             field mean + ridge on its residual  0.812  0.152     1267
Kex          mean of the field's other samples   0.693  0.129     1266
             one other sample of the field       0.547  0.157     1266
             ridge, spectra, shuffled folds      0.377  0.184     1266
             field mean + ridge on its residual  0.693  0.129     1266

## Fields with no wet chemistry

Folds grouped by field, as in `20_grouped_cv`, so that no test field has training samples. `field_baselines` gives two predictions that use no spectra, fitted on the training fields of each fold: the mean of the training samples with the same soil type and land use, falling back to the soil type and then to the overall mean, and the mean of the five training fields nearest by coordinates:

In [ ]:
#| export
def field_baselines(
    y, # Property of each sample
    folds, # Fold of each sample, grouped by field
    fields, # Field of each sample
    cat:pd.DataFrame, # Soil type and land use of each sample, as columns `Soil_type_describe` and `fieldtype`
    coords, # Latitude and longitude of each sample
    k=5, # Number of neighbouring fields
)->tuple: # Out-of-fold predictions from soil type and land use, and from the neighbouring fields
    "Out-of-fold predictions of `y` from the training samples of the same soil type and land use, and from the `k` nearest training fields"
    pc, pn = np.full(len(y), np.nan), np.full(len(y), np.nan)
    for f in np.unique(folds):
        tr, te = folds != f, folds == f
        d = cat[tr].assign(y=y[tr])
        m2, m1 = d.groupby(['Soil_type_describe', 'fieldtype']).y.mean(), d.groupby('Soil_type_describe').y.mean()
        pc[te] = [m2.get((s, l), m1.get(s, y[tr].mean())) for s, l in zip(cat.Soil_type_describe[te], cat.fieldtype[te])]
        fm = pd.DataFrame({'f': fields[tr], 'y': y[tr], 'lat': coords[tr, 0], 'lon': coords[tr, 1]}).groupby('f').mean()
        nn = NearestNeighbors(n_neighbors=k).fit(fm[['lat', 'lon']].to_numpy())
        pn[te] = fm.y.to_numpy()[nn.kneighbors(coords[te])[1]].mean(1)
    return pc, pn

In [ ]:
y_ = np.r_[np.zeros(4), np.ones(4)]
f_ = np.array(list('aabbccdd'))
cat_ = pd.DataFrame({'Soil_type_describe': ['A'] * 4 + ['B'] * 4, 'fieldtype': ['p'] * 8})
xy_ = np.c_[np.r_[0, 0, 1, 1, 10, 10, 11, 11], np.zeros(8)]
pc_, pn_ = field_baselines(y_, np.repeat([0, 1, 2, 3], 2), f_, cat_, xy_, k=1)
test_eq(pc_, y_); test_eq(pn_, y_)

Each prediction is scored against the soil type and land use mean, with 95% intervals from resampling whole fields (`sample_scores` of `20_grouped_cv`). The spectra and block 4 are used alone and with soil type and land use as extra inputs, and block 4 is also averaged with the neighbour prediction:

In [ ]:
#| fukudata
OH = pd.get_dummies(cat).to_numpy(float)
oof, scores = {}, {}
for t, y in targets.items():
    ok = np.isfinite(y)
    yy, fo = y[ok], fields[ok]
    folds = group_folds(fo)
    pc, pn = field_baselines(yy, folds, fo, cat[ok].reset_index(drop=True), coords[ok])
    P = {'soil type and land use mean': pc, 'neighbouring fields': pn}
    for k, X in {'ridge, spectra': Xd, 'ridge, block 4': B4, 'ridge, spectra + soil type and land use': np.hstack([Xd, OH]),
                 'ridge, block 4 + soil type and land use': np.hstack([B4, OH])}.items():
        P[k], _ = oof_preds(*ridge, X[ok], yy, folds, fo)
    P['mean of neighbouring fields and block 4'] = (pn + P['ridge, block 4']) / 2
    oof[t] = P
    scores[t] = sample_scores(yy, P, np.arange(len(yy)), fo, ref='soil type and land use mean')
cols = ['R²', 'R² low', 'R² high', 'RMSE', 'gain', 'gain low', 'gain high']
pd.concat(scores)[cols].round(3)

/Users/franckalbinet/aai-ws/.venv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/Users/franckalbinet/aai-ws/.venv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/Users/franckalbinet/aai-ws/.venv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/Users/franckalbinet/aai-ws/.venv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. Thi

R²  ...  gain high
Cs-137 ratio soil type and land use mean              0.078  ...      0.000
             neighbouring fields                      0.460  ...      0.490
             ridge, spectra                           0.524  ...      0.521
             ridge, block 4                           0.571  ...      0.567
             ridge, spectra + soil type and land use  0.531  ...      0.525
             ridge, block 4 + soil type and land use  0.577  ...      0.573
             mean of neighbouring fields and block 4  0.611  ...      0.618
Kex          soil type and land use mean              0.155  ...      0.000
             neighbouring fields                     -0.006  ...     -0.030
             ridge, spectra                           0.279  ...      0.207
             ridge, block 4                           0.264  ...      0.195
             ridge, spectra + soil type and land use  0.312  ...      0.233
             ridge, block 4 + soil type and land use  0.293  ...      0.218
             mean of neighbouring fields and block 4  0.249  ...      0.183

[14 rows x 7 columns]

The gain of each prediction over the neighbouring fields, for the Cs-137 ratio:

In [ ]:
#| fukudata
t = 'Cs-137 ratio'
ok = np.isfinite(targets[t])
keep = ['neighbouring fields', 'ridge, spectra', 'ridge, block 4', 'mean of neighbouring fields and block 4']
sample_scores(targets[t][ok], {k: oof[t][k] for k in keep}, np.arange(ok.sum()), fields[ok], ref='neighbouring fields')[cols].round(3)

,R²,R² low,R² high,RMSE,gain,gain low,gain high
neighbouring fields,0.460,0.351,0.549,0.257,0.000,0.000,0.000
"ridge, spectra",0.524,0.452,0.583,0.241,0.064,-0.012,0.147
"ridge, block 4",0.571,0.509,0.627,0.229,0.111,0.034,0.196
mean of neighbouring fields and block 4,0.611,0.541,0.669,0.218,0.151,0.105,0.206


## Findings

Both properties are mostly a property of the field. Across years, 85% of the variance of the Cs-137 ratio and 76% of that of Kex lies between fields. The standard deviation within a field is 0.13 for the ratio (log10) and 0.12 for `log1p(Kex)`, against 0.32 and 0.20 between fields. The sampling year explains 3% and 0.6%. Kex varies more within fields in paddy fields (66% between) than in upland fields (74%).

For fields with wet chemistry, the field's own samples beat the spectra. The mean of a field's other samples, all from other years, predicts the ratio at R² 0.81 and Kex at 0.69. One other sample of the field gives 0.68 and 0.55. Ridge on the spectra, with test samples from fields that have training samples, gives 0.58 and 0.38. Fitted to the residual from the field mean, the spectra add nothing (R² 0.81 and 0.69). If every field gets at least one wet-chemistry sample, the spectra do not improve that field's estimate. Samples of the same year would make the field mean an even harder baseline. This dataset has one sample per field and year, so it cannot test that.

For fields with no wet chemistry, the spectra beat the baselines that use no spectra. For the Cs-137 ratio, the soil type and land use mean reaches R² 0.08, and the five neighbouring fields 0.46. The coordinates are those of each field: 386 distinct pairs for 391 fields. Ridge on the spectra reaches 0.52, and on block 4 of the OSSL models 0.57. Over the neighbouring fields, block 4 gains 0.11 (95% interval 0.03 to 0.20), and the spectra 0.06 (−0.01 to 0.15). The average of the neighbour prediction and block 4 is best, at 0.61, a gain of 0.15 (0.11 to 0.21) over the neighbours. Location and spectra carry partly different information. A model fitted on both, with the neighbour feature computed within each fold, was not tried.

Kex is a property of each field, without a pattern in space or soil type. The neighbouring fields predict it at R² −0.01, and the soil type and land use mean at 0.16. The spectra reach 0.28, and 0.31 with soil type and land use as extra inputs, a gain of 0.16 (0.08 to 0.23) over their mean. A property that differs from field to field but not between neighbours or soil types fits management field by field, such as potassium fertilisation. These data do not show it.

For remediation, the spectra are useful for fields without wet chemistry, for the Cs-137 ratio together with the neighbouring fields. For fields with wet chemistry, the field's own measurements are the better estimate, and for Kex each field probably needs its own measurement or its fertilisation records.